In [2]:
# Cell 1 - Import Libraries & Pull Dell Financial Data
import yfinance as yf
import pandas as pd
import numpy as np

# Pull Dell data
company = yf.Ticker("DELL")

# Check what we have
print("=== DELL Technologies ===")
print("\n-- Income Statement --")
income_stmt = company.financials
print(income_stmt)

print("\n-- Balance Sheet --")
balance_sheet = company.balance_sheet
print(balance_sheet)

=== DELL Technologies ===

-- Income Statement --
                                                      2026-01-31  \
Tax Effect Of Unusual Items                         1.790527e+07   
Tax Rate For Calcs                                  1.827070e-01   
Normalized EBITDA                                   1.175400e+10   
Total Unusual Items                                 9.800000e+07   
Total Unusual Items Excluding Goodwill              9.800000e+07   
Net Income From Continuing Operation Net Minori...  5.936000e+09   
Reconciled Depreciation                             3.029000e+09   
Reconciled Cost Of Revenue                          9.083100e+10   
EBITDA                                              1.185200e+10   
EBIT                                                8.823000e+09   
Net Interest Income                                -1.281000e+09   
Interest Expense                                    1.560000e+09   
Interest Income                                     2.790000e+08  

In [3]:
# Cell 2 - Clean & Format the Data for Excel
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --- Extract Key Metrics ---
# Income Statement - rows we care about
income_rows = [
    "Total Revenue",
    "Gross Profit", 
    "Operating Income",
    "EBITDA",
    "Net Income",
    "Diluted EPS",
    "Research And Development",
    "Selling General And Administration"
]

# Balance Sheet - rows we care about
balance_rows = [
    "Total Debt",
    "Net Debt",
    "Cash And Cash Equivalents",
    "Accounts Receivable",
    "Total Assets" if "Total Assets" in company.balance_sheet.index else "Net Debt"
]

# Filter and clean
income_clean = income_stmt.loc[income_rows].copy()
balance_clean = balance_sheet.loc[
    [r for r in balance_rows if r in balance_sheet.index]
].copy()

# Convert to billions for readability
income_billions = (income_clean / 1e9).round(2)
balance_billions = (balance_clean / 1e9).round(2)

# Rename columns to just the year
income_billions.columns = [str(c)[:4] for c in income_billions.columns]
balance_billions.columns = [str(c)[:4] for c in balance_billions.columns]

print("=== INCOME STATEMENT (Billions $) ===")
print(income_billions)
print("\n=== BALANCE SHEET (Billions $) ===")
print(balance_billions)

=== INCOME STATEMENT (Billions $) ===
                                      2026   2025   2024    2023  2022
Total Revenue                       113.54  95.57  88.42  102.30   NaN
Gross Profit                         22.71  21.25  21.07   22.69   NaN
Operating Income                      8.45   6.66   5.93    5.77   NaN
EBITDA                               11.85   9.59   8.89    7.66   NaN
Net Income                            5.94   4.59   3.39    2.44   NaN
Diluted EPS                           0.00   0.00   0.00    0.00   NaN
Research And Development              3.14   3.06   2.80    2.78   NaN
Selling General And Administration   11.12  11.53  12.34   14.14   NaN

=== BALANCE SHEET (Billions $) ===
                             2026   2025   2024   2023  2022
Total Debt                  31.50  24.57  25.99  29.59   NaN
Net Debt                    19.98  20.93  18.63  20.98   NaN
Cash And Cash Equivalents   11.53   3.63   7.37   8.61   NaN
Accounts Receivable         17.58  10.30   

In [4]:
# Cell 3 - Build Formatted Excel Report
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# --- Color Palette ---
DARK_BLUE   = "1F3864"
MID_BLUE    = "2E75B6"
LIGHT_BLUE  = "D6E4F0"
WHITE       = "FFFFFF"
GREEN       = "E2EFDA"
GRAY        = "F2F2F2"

def style_header(cell, bg=DARK_BLUE, font_color=WHITE, size=11, bold=True):
    cell.font = Font(bold=bold, color=font_color, size=size)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center")

def style_section(cell):
    cell.font = Font(bold=True, color=WHITE, size=10)
    cell.fill = PatternFill("solid", fgColor=MID_BLUE)
    cell.alignment = Alignment(horizontal="left", vertical="center")

def add_border(cell):
    thin = Side(style="thin", color="BFBFBF")
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

# =====================
# SHEET 1 - COVER PAGE
# =====================
ws_cover = wb.active
ws_cover.title = "Cover"
ws_cover.column_dimensions["A"].width = 60

ws_cover["A1"] = "DELL TECHNOLOGIES INC."
ws_cover["A1"].font = Font(bold=True, size=24, color=DARK_BLUE)
ws_cover["A2"] = "Financial Analysis Report"
ws_cover["A2"].font = Font(size=16, color=MID_BLUE)
ws_cover["A3"] = "Fiscal Years 2023 – 2026"
ws_cover["A3"].font = Font(size=12, color="666666")
ws_cover["A4"] = ""
ws_cover["A5"] = "Prepared using Python Automated Report Generator"
ws_cover["A5"].font = Font(size=10, italic=True, color="999999")
ws_cover["A6"] = f"Data Source: Yahoo Finance  |  Currency: USD Billions"
ws_cover["A6"].font = Font(size=10, color="999999")

# =====================
# SHEET 2 - INCOME STATEMENT
# =====================
ws_inc = wb.create_sheet("Income Statement")
ws_inc.column_dimensions["A"].width = 35

years = list(income_billions.columns)

# Title
ws_inc.merge_cells("A1:F1")
ws_inc["A1"] = "DELL TECHNOLOGIES — INCOME STATEMENT (USD Billions)"
style_header(ws_inc["A1"], size=12)
ws_inc.row_dimensions[1].height = 30

# Column headers
ws_inc["A2"] = "Metric"
style_header(ws_inc["A2"], bg=MID_BLUE)
for i, yr in enumerate(years):
    col = get_column_letter(i + 2)
    ws_inc.column_dimensions[col].width = 12
    cell = ws_inc[f"{col}2"]
    cell.value = yr
    style_header(cell, bg=MID_BLUE)

# Row labels (cleaner names)
row_labels = {
    "Total Revenue": "Total Revenue",
    "Gross Profit": "Gross Profit",
    "Operating Income": "Operating Income",
    "EBITDA": "EBITDA",
    "Net Income": "Net Income",
    "Diluted EPS": "Diluted EPS",
    "Research And Development": "R&D Expense",
    "Selling General And Administration": "SG&A Expense"
}

# EPS fix - pull directly without dividing by billion
eps_row = income_stmt.loc["Diluted EPS"]

for row_idx, (key, label) in enumerate(row_labels.items()):
    excel_row = row_idx + 3
    ws_inc.row_dimensions[excel_row].height = 20

    # Section divider for EPS
    if label == "Diluted EPS":
        ws_inc.merge_cells(f"A{excel_row}:F{excel_row}")
        ws_inc[f"A{excel_row}"] = "  Per Share Data"
        style_section(ws_inc[f"A{excel_row}"])
        excel_row += 1
        ws_inc.row_dimensions[excel_row].height = 20

    # Row label
    label_cell = ws_inc[f"A{excel_row}"]
    label_cell.value = label
    label_cell.font = Font(bold=(label in ["Total Revenue", "Net Income", "EBITDA"]), size=10)
    label_cell.fill = PatternFill("solid", fgColor=GRAY if row_idx % 2 == 0 else WHITE)
    label_cell.alignment = Alignment(horizontal="left", indent=1)
    add_border(label_cell)

    # Data values
    for col_idx, yr in enumerate(years):
        col = get_column_letter(col_idx + 2)
        cell = ws_inc[f"{col}{excel_row}"]

        if label == "Diluted EPS":
            val = eps_row.get(
                [c for c in income_stmt.columns if str(c)[:4] == yr][0], None
            )
            cell.value = round(float(val), 2) if val and not pd.isna(val) else "N/A"
            cell.number_format = '"$"#,##0.00'
        else:
            val = income_billions.loc[key, yr] if yr in income_billions.columns else None
            cell.value = float(val) if val and not pd.isna(val) else "N/A"
            cell.number_format = '"$"#,##0.00"B"'

        cell.fill = PatternFill("solid", fgColor=LIGHT_BLUE if label in ["Total Revenue", "Net Income", "EBITDA"] else (GRAY if row_idx % 2 == 0 else WHITE))
        cell.alignment = Alignment(horizontal="right")
        add_border(cell)

# =====================
# SHEET 3 - BALANCE SHEET
# =====================
ws_bal = wb.create_sheet("Balance Sheet")
ws_bal.column_dimensions["A"].width = 35

ws_bal.merge_cells("A1:F1")
ws_bal["A1"] = "DELL TECHNOLOGIES — BALANCE SHEET (USD Billions)"
style_header(ws_bal["A1"], size=12)
ws_bal.row_dimensions[1].height = 30

ws_bal["A2"] = "Metric"
style_header(ws_bal["A2"], bg=MID_BLUE)
for i, yr in enumerate(years):
    col = get_column_letter(i + 2)
    ws_bal.column_dimensions[col].width = 12
    cell = ws_bal[f"{col}2"]
    cell.value = yr
    style_header(cell, bg=MID_BLUE)

bal_labels = {
    "Total Assets": "Total Assets",
    "Total Debt": "Total Debt",
    "Net Debt": "Net Debt",
    "Cash And Cash Equivalents": "Cash & Equivalents",
    "Accounts Receivable": "Accounts Receivable"
}

for row_idx, (key, label) in enumerate(bal_labels.items()):
    excel_row = row_idx + 3
    ws_bal.row_dimensions[excel_row].height = 20

    label_cell = ws_bal[f"A{excel_row}"]
    label_cell.value = label
    label_cell.font = Font(bold=(label in ["Total Assets", "Total Debt"]), size=10)
    label_cell.fill = PatternFill("solid", fgColor=GRAY if row_idx % 2 == 0 else WHITE)
    label_cell.alignment = Alignment(horizontal="left", indent=1)
    add_border(label_cell)

    for col_idx, yr in enumerate(years):
        col = get_column_letter(col_idx + 2)
        cell = ws_bal[f"{col}{excel_row}"]
        val = balance_billions.loc[key, yr] if key in balance_billions.index and yr in balance_billions.columns else None
        cell.value = float(val) if val is not None and not pd.isna(val) else "N/A"
        cell.number_format = '"$"#,##0.00"B"'
        cell.fill = PatternFill("solid", fgColor=LIGHT_BLUE if label in ["Total Assets", "Total Debt"] else (GRAY if row_idx % 2 == 0 else WHITE))
        cell.alignment = Alignment(horizontal="right")
        add_border(cell)

# =====================
# SHEET 4 - KEY METRICS
# =====================
ws_kpi = wb.create_sheet("Key Metrics")
ws_kpi.column_dimensions["A"].width = 35
ws_kpi.column_dimensions["B"].width = 20
ws_kpi.column_dimensions["C"].width = 35

ws_kpi.merge_cells("A1:C1")
ws_kpi["A1"] = "DELL TECHNOLOGIES — KEY METRICS SUMMARY"
style_header(ws_kpi["A1"], size=12)
ws_kpi.row_dimensions[1].height = 30

# Calculate KPIs using most recent year (2026)
rev_26 = income_billions.loc["Total Revenue", "2026"]
rev_25 = income_billions.loc["Total Revenue", "2025"]
gp_26  = income_billions.loc["Gross Profit", "2026"]
ni_26  = income_billions.loc["Net Income", "2026"]
ebitda_26 = income_billions.loc["EBITDA", "2026"]
debt_26 = balance_billions.loc["Total Debt", "2026"]
cash_26 = balance_billions.loc["Cash And Cash Equivalents", "2026"]

gross_margin  = (gp_26 / rev_26 * 100).round(1)
net_margin    = (ni_26 / rev_26 * 100).round(1)
ebitda_margin = (ebitda_26 / rev_26 * 100).round(1)
rev_growth    = ((rev_26 - rev_25) / rev_25 * 100).round(1)
debt_to_cash  = (debt_26 / cash_26).round(2)

kpis = [
    ("PROFITABILITY", "", ""),
    ("Revenue Growth (YoY)", f"{rev_growth}%", "↑ Strong AI-driven demand"),
    ("Gross Margin", f"{gross_margin}%", "Hardware margins typical for Dell"),
    ("EBITDA Margin", f"{ebitda_margin}%", "Improving operational efficiency"),
    ("Net Profit Margin", f"{net_margin}%", "Healthy for hardware/tech sector"),
    ("", "", ""),
    ("FINANCIAL HEALTH", "", ""),
    ("Total Debt", f"${debt_26}B", "High but manageable with cash flow"),
    ("Cash & Equivalents", f"${cash_26}B", "↑ Strong cash generation in 2026"),
    ("Debt / Cash Ratio", f"{debt_to_cash}x", "Monitor — debt exceeds cash"),
]

for row_idx, (metric, value, note) in enumerate(kpis):
    excel_row = row_idx + 2
    ws_kpi.row_dimensions[excel_row].height = 22

    if metric in ["PROFITABILITY", "FINANCIAL HEALTH"]:
        ws_kpi.merge_cells(f"A{excel_row}:C{excel_row}")
        cell = ws_kpi[f"A{excel_row}"]
        cell.value = f"  {metric}"
        style_section(cell)
    elif metric == "":
        pass
    else:
        m_cell = ws_kpi[f"A{excel_row}"]
        v_cell = ws_kpi[f"B{excel_row}"]
        n_cell = ws_kpi[f"C{excel_row}"]

        m_cell.value = metric
        m_cell.font = Font(size=10)
        m_cell.fill = PatternFill("solid", fgColor=GRAY if row_idx % 2 == 0 else WHITE)
        m_cell.alignment = Alignment(horizontal="left", indent=1)

        v_cell.value = value
        v_cell.font = Font(bold=True, size=10, color=DARK_BLUE)
        v_cell.fill = PatternFill("solid", fgColor=LIGHT_BLUE)
        v_cell.alignment = Alignment(horizontal="center")

        n_cell.value = note
        n_cell.font = Font(size=9, italic=True, color="666666")
        n_cell.fill = PatternFill("solid", fgColor=GRAY if row_idx % 2 == 0 else WHITE)
        n_cell.alignment = Alignment(horizontal="left", indent=1)

        for cell in [m_cell, v_cell, n_cell]:
            add_border(cell)

# --- Save Excel ---
excel_path = "DELL_Financial_Report.xlsx"
wb.save(excel_path)
print(f"✅ Excel report saved: {excel_path}")
print(f"   Sheets: Cover | Income Statement | Balance Sheet | Key Metrics")

✅ Excel report saved: DELL_Financial_Report.xlsx
   Sheets: Cover | Income Statement | Balance Sheet | Key Metrics


In [5]:
# Cell 4 - Open the Excel file so we can see it
import subprocess
subprocess.Popen(["start", "excel", "DELL_Financial_Report.xlsx"], shell=True)

<Popen: returncode: None args: ['start', 'excel', 'DELL_Financial_Report.xlsx']>

In [6]:
# Cell 5 - Generate PDF Report
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.enums import TA_CENTER, TA_LEFT

# --- Setup ---
pdf_path = "DELL_Financial_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=letter,
                        rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

# --- Colors ---
DARK_BLUE  = colors.HexColor("#1F3864")
MID_BLUE   = colors.HexColor("#2E75B6")
LIGHT_BLUE = colors.HexColor("#D6E4F0")
GRAY       = colors.HexColor("#F2F2F2")
WHITE      = colors.white

# --- Styles ---
styles = getSampleStyleSheet()

title_style = ParagraphStyle("title",
    fontSize=22, textColor=DARK_BLUE,
    alignment=TA_CENTER, fontName="Helvetica-Bold", spaceAfter=6)

subtitle_style = ParagraphStyle("subtitle",
    fontSize=13, textColor=MID_BLUE,
    alignment=TA_CENTER, fontName="Helvetica", spaceAfter=4)

meta_style = ParagraphStyle("meta",
    fontSize=9, textColor=colors.HexColor("#999999"),
    alignment=TA_CENTER, fontName="Helvetica-Oblique", spaceAfter=20)

section_style = ParagraphStyle("section",
    fontSize=13, textColor=DARK_BLUE,
    fontName="Helvetica-Bold", spaceAfter=8, spaceBefore=16)

note_style = ParagraphStyle("note",
    fontSize=8, textColor=colors.HexColor("#666666"),
    fontName="Helvetica-Oblique", spaceAfter=4)

# --- Helper: Table Style ---
def make_table_style(num_rows):
    style = [
        # Header row
        ("BACKGROUND",    (0,0), (-1,0), DARK_BLUE),
        ("TEXTCOLOR",     (0,0), (-1,0), WHITE),
        ("FONTNAME",      (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE",      (0,0), (-1,0), 9),
        ("ALIGN",         (0,0), (-1,0), "CENTER"),
        ("BOTTOMPADDING", (0,0), (-1,0), 8),
        ("TOPPADDING",    (0,0), (-1,0), 8),
        # Data rows
        ("FONTNAME",      (0,1), (-1,-1), "Helvetica"),
        ("FONTSIZE",      (0,1), (-1,-1), 9),
        ("ALIGN",         (1,1), (-1,-1), "RIGHT"),
        ("ALIGN",         (0,1), (0,-1), "LEFT"),
        ("LEFTPADDING",   (0,1), (0,-1), 8),
        ("RIGHTPADDING",  (1,1), (-1,-1), 8),
        ("BOTTOMPADDING", (0,1), (-1,-1), 5),
        ("TOPPADDING",    (0,1), (-1,-1), 5),
        ("GRID",          (0,0), (-1,-1), 0.5, colors.HexColor("#BFBFBF")),
    ]
    # Alternating row colors
    for i in range(1, num_rows):
        bg = LIGHT_BLUE if i % 2 == 0 else WHITE
        style.append(("BACKGROUND", (0,i), (-1,i), bg))
    return TableStyle(style)

# --- Build Content ---
content = []

# Cover
content.append(Spacer(1, 0.5*inch))
content.append(Paragraph("DELL TECHNOLOGIES INC.", title_style))
content.append(Paragraph("Financial Analysis Report", subtitle_style))
content.append(Paragraph("Fiscal Years 2023 – 2026  |  USD Billions  |  Source: Yahoo Finance", meta_style))
content.append(Spacer(1, 0.2*inch))

# --- Section 1: Income Statement ---
content.append(Paragraph("1. Income Statement", section_style))

inc_headers = ["Metric", "FY2023", "FY2024", "FY2025", "FY2026"]
inc_data = [inc_headers]

inc_rows_pdf = [
    ("Total Revenue",    "Total Revenue"),
    ("Gross Profit",     "Gross Profit"),
    ("Operating Income", "Operating Income"),
    ("EBITDA",           "EBITDA"),
    ("Net Income",       "Net Income"),
    ("Research And Development", "R&D Expense"),
    ("Selling General And Administration", "SG&A Expense"),
]

for key, label in inc_rows_pdf:
    row = [label]
    for yr in ["2023", "2024", "2025", "2026"]:
        val = income_billions.loc[key, yr] if yr in income_billions.columns else None
        row.append(f"${val:.2f}B" if val is not None and not pd.isna(val) else "N/A")
    inc_data.append(row)

inc_table = Table(inc_data, colWidths=[2.2*inch, 1*inch, 1*inch, 1*inch, 1*inch])
inc_table.setStyle(make_table_style(len(inc_data)))
content.append(inc_table)

# --- Section 2: Balance Sheet ---
content.append(Paragraph("2. Balance Sheet", section_style))

bal_headers = ["Metric", "FY2023", "FY2024", "FY2025", "FY2026"]
bal_data = [bal_headers]

bal_rows_pdf = [
    ("Total Assets",               "Total Assets"),
    ("Total Debt",                 "Total Debt"),
    ("Net Debt",                   "Net Debt"),
    ("Cash And Cash Equivalents",  "Cash & Equivalents"),
    ("Accounts Receivable",        "Accounts Receivable"),
]

for key, label in bal_rows_pdf:
    row = [label]
    for yr in ["2023", "2024", "2025", "2026"]:
        val = balance_billions.loc[key, yr] if key in balance_billions.index and yr in balance_billions.columns else None
        row.append(f"${val:.2f}B" if val is not None and not pd.isna(val) else "N/A")
    bal_data.append(row)

bal_table = Table(bal_data, colWidths=[2.2*inch, 1*inch, 1*inch, 1*inch, 1*inch])
bal_table.setStyle(make_table_style(len(bal_data)))
content.append(bal_table)

# --- Section 3: Key Metrics ---
content.append(Paragraph("3. Key Performance Metrics (FY2026)", section_style))

kpi_data = [
    ["Metric", "Value", "Insight"],
    ["Revenue Growth (YoY)",  f"{rev_growth}%",      "AI-driven server demand surge"],
    ["Gross Margin",          f"{gross_margin}%",     "Typical for hardware/tech mix"],
    ["EBITDA Margin",         f"{ebitda_margin}%",    "Improving operational leverage"],
    ["Net Profit Margin",     f"{net_margin}%",       "Healthy for sector"],
    ["Cash & Equivalents",    f"${cash_26}B",         "Strong cash generation in FY2026"],
    ["Total Debt",            f"${debt_26}B",         "High but declining trend expected"],
    ["Debt / Cash Ratio",     f"{debt_to_cash}x",     "Monitor leverage closely"],
]

kpi_table = Table(kpi_data, colWidths=[2.2*inch, 1*inch, 3*inch])
kpi_table.setStyle(make_table_style(len(kpi_data)))
content.append(kpi_table)

# --- Section 4: Analyst Commentary ---
content.append(Paragraph("4. Analyst Commentary", section_style))

commentary = [
    "<b>Revenue Growth:</b> Dell's revenue surged 18.8% YoY in FY2026, driven primarily by explosive demand for AI servers and infrastructure. This follows a dip in FY2024 as enterprise spending normalized post-pandemic.",
    "<b>Profitability Trend:</b> Net income has nearly tripled from $2.4B in FY2023 to $5.9B in FY2026, reflecting both revenue growth and disciplined cost reduction — SG&A dropped from $14.1B to $11.1B over the same period.",
    "<b>Balance Sheet:</b> Cash jumped from $3.6B to $11.5B in FY2026 — a strong signal of improving free cash flow. Total debt remains elevated at $31.5B but is being actively managed.",
    "<b>Risk Factors:</b> Margins remain thin for a company of this scale (20% gross margin). Dell's growth is heavily tied to AI infrastructure spending, which could slow if enterprise capex contracts.",
]

for point in commentary:
    content.append(Paragraph(f"• {point}", styles["Normal"]))
    content.append(Spacer(1, 0.1*inch))

content.append(Spacer(1, 0.2*inch))
content.append(Paragraph("* This report was generated programmatically using Python and Yahoo Finance data. Not investment advice.", note_style))

# --- Build PDF ---
doc.build(content)
print(f"✅ PDF report saved: {pdf_path}")

✅ PDF report saved: DELL_Financial_Report.pdf


In [7]:
# Cell 6 - Open the PDF
import subprocess
subprocess.Popen(["start", "", "DELL_Financial_Report.pdf"], shell=True)

<Popen: returncode: None args: ['start', '', 'DELL_Financial_Report.pdf']>